# Dispersion fast paths: parity with the reference (DFT-D3, DFT-D4)

xnn's D3 and D4 terms have two implementations, selected per evaluation with `use_fast`
(see the *Run Models on Fused GPU Kernels* how-to):

* the **reference** code, verified against `simple-dftd3` and `dftd4` in the fidelity notebooks;
* the **fast paths** of `xnn.common.models.dispersion_fast`: the Axilrod-Teller-Muto three-body
  term as fused Triton kernels (forward and closed-form gradient, the pair $C_6$ from per-atom
  factors, no triplet ever stored), and D4's large-system EEQ solve with a factorized Ewald
  operator (no trigonometry in the conjugate-gradient loop, a sparse real-space matrix, the
  charge and constraint solves in lockstep).

This notebook checks that the two agree, on the systems D4 was developed and profiled on:
periodic boxes of liquid water at about 1 g/cm$^3$ with the settings served next to an MLIP
(pair cutoff 12 Å with a 2 Å switching window, three-body cutoff 8 Å with a 1 Å window,
coordination numbers to 12 Å), evaluated for energy, forces and stress. A water cluster
(molecular) and a cell narrower than twice the three-body cutoff (periodic self-image
triangles, a separate code path of the kernels) complete the set.

What to expect: the three-body kernels compute the same sums in another order, so in float64
they agree with the reference to rounding (about $10^{-14}$ relative). The fast EEQ operator is
the same matrix, but the solves are iterative, so the charges agree to the solver tolerance
(about $10^{-12}$ in float64). In float32 both are limited by single precision (about
$10^{-6}$ relative).

In [1]:
import warnings

warnings.filterwarnings("ignore")          # TorchScript / deprecation chatter of the stack

import copy
import time

import numpy as np
import torch

from xnn.common.data import structure_to_graph
from xnn.common.models import D3Dispersion, D4Dispersion, ForceStressOutput
from xnn.common.models import fast

assert torch.cuda.is_available(), "the fast paths run on a CUDA device"
device = torch.device("cuda")
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

NVIDIA A100-SXM4-80GB | torch 2.5.1+cu121


## 1. Systems and helpers

A jittered lattice of randomly oriented waters (3.104 Å apart, 1 g/cm$^3$), the same
construction as `tools/d4_bench.py`, and the served D3/D4 settings. `run` evaluates a model
with the reference or the fast paths (after one warm-up evaluation, which compiles the
kernels) and times one energy + forces + stress evaluation; `differences` reports the largest
deviations of the fast result from the reference.

In [2]:
WATER = np.array([[0.0, 0.0, 0.119262], [0.0, 0.763239, -0.477047], [0.0, -0.763239, -0.477047]])
SERVED = dict(cutoff_pair=12.0, switch_width_pair=2.0, cutoff_triple=8.0, switch_width_triple=1.0,
              s9=1.0, cutoff_cn=12.0)
D4_EXTRA = dict(cutoff_eeq_cn=12.0)


def water(n_side, periodic=True, seed=7):
    rng = np.random.default_rng(seed)
    pos = []
    for i in range(n_side):
        for j in range(n_side):
            for k in range(n_side):
                q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
                q *= np.sign(np.linalg.det(q))
                pos.extend((WATER - WATER[0]) @ q.T + np.array([i, j, k]) * 3.104
                           + rng.normal(scale=0.1, size=3))
    s = {"pos": np.array(pos), "atomic_numbers": [8, 1, 1] * n_side ** 3}
    if periodic:
        s["cell"] = np.eye(3) * 3.104 * n_side
        s["pbc"] = [True, True, True]
    return s


def model_of(method, dtype, **options):
    cls = D4Dispersion if method == "d4" else D3Dispersion
    opts = {**SERVED, **(D4_EXTRA if method == "d4" else {}), **options}
    return ForceStressOutput(cls(**opts), compute_stress=True).to(device=device, dtype=dtype).eval()


def graph_of(s, model, dtype):
    s = {k: (torch.tensor(v, dtype=dtype) if k in ("pos", "cell") else v) for k, v in s.items()}
    return structure_to_graph(s, model.model.cutoff, device=device)


def run(model, g, use_fast):
    fast.set_use_fast(model, use_fast)
    model(copy.copy(g))                                   # warm-up
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model(copy.copy(g))
    torch.cuda.synchronize()
    return {k: v.detach() for k, v in out.items()}, time.perf_counter() - t0


def differences(out, ref):
    rel = lambda a, b: float((a - b).abs().max() / b.abs().max().clamp(min=1e-300))
    d = {"dE/E": rel(out["energy"], ref["energy"]),
         "dE3/E3": rel(out["energy_3body"], ref["energy_3body"]),
         "dF max (eV/A)": float((out["forces"] - ref["forces"]).abs().max()),
         "dF/|F|max": rel(out["forces"], ref["forces"])}
    if "stress" in ref and ref["stress"].abs().max() > 0:
        d["dS/|S|max"] = rel(out["stress"], ref["stress"])
    if "eeq_charges" in ref:
        d["dq max (e)"] = float((out["eeq_charges"] - ref["eeq_charges"]).abs().max())
    return d


def report(label, ref, out, t_ref, t_fast):
    d = differences(out, ref)
    print(f"{label:34s} ref {t_ref * 1e3:8.1f} ms | fast {t_fast * 1e3:7.1f} ms "
          f"({t_ref / t_fast:5.1f}x) | " + "  ".join(f"{k} {v:.1e}" for k, v in d.items()))
    return d

## 2. 1728 waters (5184 atoms), the D4 development box

At this size D4 is in its large EEQ regime (above 1500 atoms in a cell): the reference
factorizes the assembled EEQ matrix, the fast path solves with conjugate gradients on the
factorized operator. The three-body term covers about 40 million triangles.

In [3]:
results = {}
s5k = water(12)
for method in ("d4", "d3"):
    for dtype in (torch.float64, torch.float32):
        model = model_of(method, dtype)
        g = graph_of(s5k, model, dtype)
        ref, t_ref = run(model, g, False)
        out, t_fast = run(model, g, True)
        label = f"{method.upper()} {str(dtype)[6:]} 5184 atoms"
        results[label] = report(label, ref, out, t_ref, t_fast)
        del model, g, ref, out
        torch.cuda.empty_cache()

D4 float64 5184 atoms              ref    596.0 ms | fast   229.3 ms (  2.6x) | dE/E 5.2e-15  dE3/E3 4.9e-16  dF max (eV/A) 5.5e-13  dF/|F|max 4.2e-11  dS/|S|max 9.6e-15  dq max (e) 6.8e-12


D4 float32 5184 atoms              ref    467.2 ms | fast   145.6 ms (  3.2x) | dE/E 5.6e-07  dE3/E3 3.5e-07  dF max (eV/A) 9.7e-07  dF/|F|max 7.4e-05  dS/|S|max 6.8e-07  dq max (e) 1.0e-05


D3 float64 5184 atoms              ref    490.7 ms | fast    71.4 ms (  6.9x) | dE/E 1.9e-16  dE3/E3 1.4e-15  dF max (eV/A) 3.3e-17  dF/|F|max 6.4e-15  dS/|S|max 1.4e-16


D3 float32 5184 atoms              ref    367.4 ms | fast    38.1 ms (  9.6x) | dE/E 1.0e-07  dE3/E3 8.1e-07  dF max (eV/A) 2.8e-08  dF/|F|max 5.4e-06  dS/|S|max 3.0e-07


On the fast path D3 does not form its dense `(N, N)` C6 matrix (the three-body kernels read
the pair $C_6$ from per-atom factors), so the `"c6_matrix"` output is empty there;
`DFTD3.c6_matrix` forms it on request.

## 3. Larger boxes: the iterative EEQ solve, and D3 past its old limit

At 4096 waters (12 288 atoms) both implementations solve the EEQ system with conjugate
gradients, which is where the reference spent 85-90% of a D4 step recomputing structure
factors. At 8000 waters (24 000 atoms) the reference D3 three-body term refuses (it needs the
dense C6 matrix, at most 20 000 atoms); the fast path has no such limit, so only its time is
shown.

In [4]:
model = model_of("d4", torch.float64)
g = graph_of(water(16), model, torch.float64)
ref, t_ref = run(model, g, False)
out, t_fast = run(model, g, True)
results["D4 float64 12288 atoms"] = report("D4 float64 12288 atoms", ref, out, t_ref, t_fast)
del model, g, ref, out
torch.cuda.empty_cache()

model = model_of("d3", torch.float32)
g = graph_of(water(20), model, torch.float32)
out, t_fast = run(model, g, True)
print(f"D3 float32 24000 atoms: fast {t_fast * 1e3:.1f} ms, E_3body = {float(out['energy_3body']):.6f} eV "
      f"(the reference raises above 20000 atoms)")
del model, g, out
torch.cuda.empty_cache()

D4 float64 12288 atoms             ref  11539.8 ms | fast   467.1 ms ( 24.7x) | dE/E 4.4e-15  dE3/E3 3.5e-15  dF max (eV/A) 2.4e-12  dF/|F|max 1.8e-10  dS/|S|max 7.2e-15  dq max (e) 2.8e-11


D3 float32 24000 atoms: fast 157.9 ms, E_3body = 35.417084 eV (the reference raises above 20000 atoms)


## 4. Self-image triangles and a cluster

In a cell narrower than twice the three-body cutoff (here 9.3 Å against 8 Å), a triangle can
contain an atom and its own periodic image; the reference counts those once per visit, and the
kernels handle them in a separate launch. A cluster has no cell at all. Both in float64.

In [5]:
for name, s in (("small cell (81 atoms, 9.3 A)", water(3)), ("cluster (192 atoms)", water(4, periodic=False))):
    for method in ("d4", "d3"):
        model = model_of(method, torch.float64)
        g = graph_of(s, model, torch.float64)
        ref, t_ref = run(model, g, False)
        out, t_fast = run(model, g, True)
        results[f"{method.upper()} {name}"] = report(f"{method.upper()} {name}", ref, out, t_ref, t_fast)

D4 small cell (81 atoms, 9.3 A)    ref     27.6 ms | fast    23.2 ms (  1.2x) | dE/E 2.1e-16  dE3/E3 4.9e-16  dF max (eV/A) 4.3e-18  dF/|F|max 4.3e-16  dS/|S|max 1.0e-17  dq max (e) 0.0e+00
D3 small cell (81 atoms, 9.3 A)    ref     20.3 ms | fast    11.6 ms (  1.7x) | dE/E 0.0e+00  dE3/E3 2.3e-16  dF max (eV/A) 6.1e-18  dF/|F|max 1.9e-15  dS/|S|max 2.8e-16
D4 cluster (192 atoms)             ref     20.5 ms | fast    14.4 ms (  1.4x) | dE/E 2.7e-16  dE3/E3 1.7e-16  dF max (eV/A) 7.8e-16  dF/|F|max 3.5e-14  dq max (e) 0.0e+00


D3 cluster (192 atoms)             ref     16.3 ms | fast     8.4 ms (  1.9x) | dE/E 0.0e+00  dE3/E3 2.1e-16  dF max (eV/A) 6.9e-18  dF/|F|max 4.7e-16


## 5. Molecular dynamics: EEQ reuse between steps

For MD, D4 can carry the EEQ solve from one step to the next (`enable_eeq_reuse`): a
preconditioner from an earlier step and extrapolated initial guesses. With the fast paths, the
remaining cost of a step is mostly the three-body term. Ten displaced frames (0.01 Å rms per
step) of the 5184-atom box in float32, against a fresh reference solve of each frame.

In [6]:
model = model_of("d4", torch.float32)
model.model.d4.enable_eeq_reuse()
reference = model_of("d4", torch.float32)
fast.set_use_fast(model, True)
fast.set_use_fast(reference, False)
rng = np.random.default_rng(11)
s = water(12)
times, dq, df = [], [], []
for frame in range(11):
    s = {**s, "pos": s["pos"] + rng.normal(scale=0.01, size=s["pos"].shape)}
    g = graph_of(s, model, torch.float32)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model(copy.copy(g))
    torch.cuda.synchronize()
    if frame > 0:                                      # frame 0 forms the preconditioner
        times.append(time.perf_counter() - t0)
    ref = reference(copy.copy(g))
    dq.append(float((out["eeq_charges"] - ref["eeq_charges"]).abs().max()))
    df.append(float((out["forces"] - ref["forces"]).abs().max()))
print(f"fast + EEQ reuse: {1e3 * np.median(times):.1f} ms per step (median of {len(times)}), "
      f"max |dq| {max(dq):.1e} e, max |dF| {max(df):.1e} eV/A against a fresh reference solve")
del model, reference
torch.cuda.empty_cache()

fast + EEQ reuse: 104.3 ms per step (median of 10), max |dq| 3.7e-05 e, max |dF| 5.4e-06 eV/A against a fresh reference solve


## 6. Second derivatives

The kernels are differentiable once. A training step (a force loss needs
`create_graph=True`) uses the reference three-body term. Outside training, a backward pass
with `create_graph=True` (a Hessian) hands the kernels' gradient to the reference, so the
result is exact. A Hessian-vector product $\partial (F \cdot v) / \partial r$ through the
dispersion term in evaluation mode, where the kernels run the forward pass:

In [7]:
for method in ("d4", "d3"):
    model = model_of(method, torch.float64)
    disp = model.model                                 # the dispersion module, in evaluation mode
    hv = {}
    for use_fast in (False, True):
        fast.set_use_fast(disp, use_fast)
        g = graph_of(water(4), model, torch.float64)
        g.pos.requires_grad_(True)
        energy = disp(g)["energy"].sum()
        forces = -torch.autograd.grad(energy, g.pos, create_graph=True)[0]
        # a random direction: the forces sum to zero, so a uniform one gives Hv = 0
        v = torch.randn(forces.shape, generator=torch.Generator().manual_seed(0), dtype=forces.dtype)
        hv[use_fast] = torch.autograd.grad((forces * v.to(device)).sum(), g.pos)[0]
    err = float((hv[True] - hv[False]).abs().max() / hv[False].abs().max())
    print(f"{method.upper()}: Hessian-vector product, max |Hv| {float(hv[False].abs().max()):.2e}, "
          f"fast vs reference: {err:.1e} relative")

D4: Hessian-vector product, max |Hv| 2.69e-01, fast vs reference: 5.9e-14 relative
D3: Hessian-vector product, max |Hv| 6.00e-02, fast vs reference: 5.1e-15 relative


## 7. Summary

Largest relative deviations of the fast paths from the reference, per system:

In [8]:
keys = sorted({k for d in results.values() for k in d})
print(f"{'system':36s}" + "".join(f"{k:>16s}" for k in keys))
for label, d in results.items():
    print(f"{label:36s}" + "".join(f"{d[k]:16.1e}" if k in d else f"{'':>16s}" for k in keys))

system                                          dE/E          dE3/E3   dF max (eV/A)       dF/|F|max       dS/|S|max      dq max (e)
D4 float64 5184 atoms                        5.2e-15         4.9e-16         5.5e-13         4.2e-11         9.6e-15         6.8e-12
D4 float32 5184 atoms                        5.6e-07         3.5e-07         9.7e-07         7.4e-05         6.8e-07         1.0e-05
D3 float64 5184 atoms                        1.9e-16         1.4e-15         3.3e-17         6.4e-15         1.4e-16                
D3 float32 5184 atoms                        1.0e-07         8.1e-07         2.8e-08         5.4e-06         3.0e-07                
D4 float64 12288 atoms                       4.4e-15         3.5e-15         2.4e-12         1.8e-10         7.2e-15         2.8e-11
D4 small cell (81 atoms, 9.3 A)              2.1e-16         4.9e-16         4.3e-18         4.3e-16         1.0e-17         0.0e+00
D3 small cell (81 atoms, 9.3 A)              0.0e+00         2.3e-16 